# BattMo.jl Workshop — Hands-on Session 2: Case Solving

**Agenda slot:** 13:00 – 14:30 (90 minutes), followed by a break and a "Discuss findings" session at 15:00.

This afternoon you'll work in teams to design a battery cell that meets a client's brief, using the same BattMo.jl workflows you practiced this morning (loading parameters, modifying them, running simulations, and reading outputs). There is no single correct answer — different teams will likely arrive at different designs. Bring your results to the 15:00 discussion.

## The brief: GreenWheels Mobility

GreenWheels Mobility is launching a new electric cargo bike, built around a cylindrical cell using the same **NMC811 / Graphite-SiOx** chemistry as the `chen_2020` parameter set you used this morning.

Their engineering team already built a first prototype. Early testing flagged a safety problem — the negative electrode wasn't sized to fully absorb the positive electrode's capacity — so they thickened the negative electrode coating to fix it. That fix worked, but it came with a side effect they haven't resolved yet: the cell can no longer deliver power at the rate the bike needs. **Your job is to close that gap without undoing the safety fix.**

You are free to change electrode coating thicknesses, particle radii, mass fractions, or other `CellParameters` entries. You may **not** change the cell format (cylindrical, same outer dimensions) or the active materials themselves (still NMC811 / Graphite-SiOx).

### Requirements

| # | Requirement | Target | Why it matters |
|---|---|---|---|
| 1 | **Safety** — N:P ratio | between **1.00 and 1.15** | Below 1.0 risks lithium plating on the negative electrode; too far above 1.0 wastes positive electrode capacity. |
| 2 | **Energy** — specific energy at 0.5C discharge | **≥ 380 Wh/kg** | Determines the bike's range for a given pack weight. |
| 3 | **Power** — capacity retention at 2C vs. 0.5C discharge | **≥ 90 %** | The bike must keep delivering power on hills without the cell "falling off a cliff". |
| 4 | **Fast charge** — a 3C CCCV charge cycle | must **complete** (no solver failure) | Riders want to top up during a lunch break. |

Your task: starting from GreenWheels' current prototype, find **one design that satisfies all four requirements at once**. Some fixes for one requirement will hurt another — that tension is the point of the exercise. Don't expect to get there in one move: expect to sweep a parameter across several values before you find one that actually clears the bar.


## Setup

Let's import the packages we'll need. These are the same ones you used this morning.

In [ ]:
using BattMo, GLMakie, Jutul

## Step 1 — Check the baseline

Let's load the `chen_2020` chemistry and apply GreenWheels' safety fix: a thicker negative electrode coating (94 µm instead of the original 85.2 µm). This is the prototype your team inherits.

In [ ]:
cell_parameters = load_cell_parameters(; from_default_set = "chen_2020")
cell_parameters["NegativeElectrode"]["Coating"]["Thickness"] = 9.4e-5  # GreenWheels' safety fix

quick_cell_check(cell_parameters)

Take a look at the **Cell N:P Ratio** — the safety fix did its job, this is now in spec. Keep this number in mind: it's easy to break again. We'll quantify all four requirements together in the next step.

## Step 2 — Build a scorecard

Checking all four requirements by hand after every change would be slow. Let's write a small helper function, `evaluate_design`, that takes a `CellParameters` object, runs everything needed to check the brief, and prints a scorecard. You'll reuse this function for the rest of the session — just call `evaluate_design(my_design)` after every change you make.

It uses functions you already know (`compute_cell_mass`, `compute_np_ratio`, `Simulation`, `solve`) plus two new ones, `compute_discharge_capacity`-style metrics already available on `output.metrics`, and a `try`/`catch` because at very high C-rates the solver can fail outright rather than just returning an empty result.

In [ ]:
function evaluate_design(cp; label = "design", verbose = true)

    mass_kg = compute_cell_mass(cp)
    np_ratio = compute_np_ratio(cp)

    model = LithiumIonBattery()

    # --- Energy: 0.5C discharge ---
    discharge_protocol = load_cycling_protocol(; from_default_set = "cc_discharge")
    discharge_protocol["DRate"] = 0.5
    sim_05C = Simulation(model, cp, discharge_protocol)
    out_05C = solve(sim_05C; info_level = -1)
    cap_05C = out_05C.metrics["DischargeCapacity"][1]
    energy_05C_Wh = out_05C.metrics["DischargeEnergy"][1] / 3600
    specific_energy = energy_05C_Wh / mass_kg

    # --- Power: 2C discharge, compared to 0.5C ---
    discharge_protocol["DRate"] = 2.0
    sim_2C = Simulation(model, cp, discharge_protocol)
    out_2C = solve(sim_2C; info_level = -1)
    cap_2C = out_2C.metrics["DischargeCapacity"][1]
    retention_2C = 100 * cap_2C / cap_05C

    # --- Fast charge: does a 3C CCCV cycle even complete? ---
    fast_charge_ok = false
    try
        cccv = load_cycling_protocol(; from_default_set = "cccv")
        cccv["CRate"] = 3.0
        cccv["TotalNumberOfCycles"] = 1
        sim_fc = Simulation(model, cp, cccv)
        out_fc = solve(sim_fc; info_level = -1)
        fast_charge_ok = length(out_fc.states["Time"]) > 0
    catch
        fast_charge_ok = false
    end

    checks = [
        ("1. Safety       (1.00 <= N:P <= 1.15)  ", 1.00 <= np_ratio <= 1.15, "N:P = $(round(np_ratio, digits = 3))"),
        ("2. Energy       (specific energy >= 380 Wh/kg)", specific_energy >= 380, "$(round(specific_energy, digits = 1)) Wh/kg"),
        ("3. Power        (2C retention >= 90%)  ", retention_2C >= 90, "$(round(retention_2C, digits = 1)) %"),
        ("4. Fast charge  (3C CCCV completes)     ", fast_charge_ok, fast_charge_ok ? "completes" : "fails"),
    ]

    if verbose
        println("="^58)
        println(" Scorecard: $label")
        println("="^58)
        for (name, passed, value) in checks
            mark = passed ? "PASS" : "FAIL"
            println(" [$mark] $name -> $value")
        end
        n_passed = count(c -> c[2], checks)
        println("-"^58)
        println(" $n_passed / 4 requirements met")
        println("="^58)
    end

    return (
        mass_kg = mass_kg,
        np_ratio = np_ratio,
        specific_energy = specific_energy,
        retention_2C = retention_2C,
        fast_charge_ok = fast_charge_ok,
        checks = checks,
    )
end

Let's run it on the prototype baseline:

In [ ]:
baseline_result = evaluate_design(cell_parameters; label = "GreenWheels prototype")

Three checks pass already — safety, energy, and fast charge. Power is the one that's failing, and not by a little. That's the gap you need to close.

## Step 3 — Visualize the rate-capability cliff

Numbers in a table only tell part of the story. Let's sweep over discharge rates and plot capacity vs. D-rate for the prototype baseline, the same way you did in this morning's assignment.

In [ ]:
d_rates = [0.2, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
capacities = Float64[]

discharge_protocol = load_cycling_protocol(; from_default_set = "cc_discharge")
model = LithiumIonBattery()

for d_rate in d_rates
    discharge_protocol["DRate"] = d_rate
    sim = Simulation(model, cell_parameters, discharge_protocol)
    output = solve(sim; info_level = -1)
    if length(output.states["Time"]) > 0
        push!(capacities, output.metrics["DischargeCapacity"][1])
    else
        push!(capacities, 0.0)
    end
end

f = Figure(size = (700, 400))
ax = Axis(f[1, 1], title = "Baseline rate capability", xlabel = "D-rate / C", ylabel = "Discharge capacity / Ah")
scatterlines!(ax, d_rates, capacities; linewidth = 4)
f

In [ ]:
GLMakie.closeall()

## Step 4 — Your design challenge

Now it's your turn. Make a working copy of the cell parameters with `deepcopy` (so you can always go back to the baseline), change one or more parameters, and call `evaluate_design` to see where you stand. Iterate until all four checks pass.

Some parameters you could explore — this is not an exhaustive list, and not every knob listed here is equally useful:

| Parameter | Location in `CellParameters` |
|---|---|
| Negative electrode coating thickness | `["NegativeElectrode"]["Coating"]["Thickness"]` |
| Positive electrode coating thickness | `["PositiveElectrode"]["Coating"]["Thickness"]` |
| Negative electrode particle radius | `["NegativeElectrode"]["ActiveMaterial"]["ParticleRadius"]` |
| Positive electrode particle radius | `["PositiveElectrode"]["ActiveMaterial"]["ParticleRadius"]` |
| Negative/Positive reaction rate constant | `[...]["ActiveMaterial"]["ReactionRateConstant"]` |

A few hints, without giving away the answer:
- Don't expect the first value you try to clear the 90% bar — sweep a parameter across a few values (e.g. half, then half again) and watch how the retention number moves before you commit to one.
- Not every parameter that sounds rate-related actually helps. Test your assumption with `evaluate_design` rather than trusting intuition alone — one of the "obvious" levers in the table above barely moves the needle, or even makes things worse.
- The N:P ratio depends on the *relative* capacity of the two electrodes — undoing the thickness fix (even slightly) to chase another requirement will put safety back out of spec. Re-run `evaluate_design` after every change.
- `quick_cell_check(cell_parameters, cell_2 = my_design)` is a fast way to compare mass/capacity/N:P side by side without running a full simulation, useful for quick sanity checks before you spend time on a full `evaluate_design` call.

### Why these parameters move the needle

Each parameter in the table above is a stand-in for something a cell manufacturer actually controls on the production line. Knowing the physical lever behind each number helps you reason about *why* a change works in the model — and what it would cost to do in practice.

| Parameter | What it controls in the model | Performance factor(s) it touches | How it's tuned in a real cell |
|---|---|---|---|
| Electrode coating thickness | Amount of active material (capacity) per unit area, and the distance lithium ions must travel through the electrode | **Energy** ↑ with thickness; **Power**/fast-charge ↓ with thickness (longer diffusion path); **N:P ratio** depends on the NE/PE capacities *relative* to each other | Slot-die coating gap (wet-film thickness) during electrode manufacturing |
| Particle radius | Length of the solid-state diffusion path inside each active material particle | **Power** ↑ and **fast-charge** feasibility ↑ as radius ↓ (shorter diffusion path) | Active material powder grade selected from the supplier (particle size distribution), or post-synthesis milling |
| Reaction rate constant | Speed of the lithium intercalation/deintercalation reaction at the particle surface | **Power** and **fast-charge** feasibility — sluggish kinetics show up as extra overpotential that gets worse at high current | Particle surface coatings (e.g. carbon coating), electrolyte/additive formulation, conductive-carbon content in the electrode, formation/conditioning protocol |
| Mass fractions (active material vs. binder/conductive carbon) | Active material loading per unit volume, and the porosity left over for electrolyte/ion transport | **Energy** ↑ with active material fraction; **Power** ↓ if porosity gets too low (higher tortuosity for ion transport) | Electrode slurry recipe, and calendering pressure (compaction), which trades porosity for energy density |

A pattern worth noticing: almost every lever that buys **energy** (more or denser active material) does so by lengthening the path lithium ions have to travel, which is exactly what hurts **power** and **fast-charge**. That tension isn't an artifact of the model — it's the same trade-off cell engineers face on the factory floor, which is why "fixing one requirement breaks another" here.


In [ ]:
my_design = deepcopy(cell_parameters)

# --- make your changes here ---


# -------------------------------

quick_cell_check(cell_parameters, cell_2 = my_design)

In [ ]:
evaluate_design(my_design; label = "My design - v1")

Keep iterating — duplicate the cell below as many times as you need (v2, v3, ...).

In [ ]:
# my_design = deepcopy(cell_parameters)
# ...
# evaluate_design(my_design; label = "My design - v2")

## Step 5 — Lock in your final design

Once your design passes all four checks, let's take a closer look at it and save it so you can bring it to the 15:00 discussion.

In [ ]:
final_result = evaluate_design(my_design; label = "Final design")

In [ ]:
model = LithiumIonBattery()
discharge_protocol = load_cycling_protocol(; from_default_set = "cc_discharge")
discharge_protocol["DRate"] = 0.5

sim_final = Simulation(model, my_design, discharge_protocol)
output_final = solve(sim_final)

plot_dashboard(output_final; plot_type = "line")

In [ ]:
GLMakie.closeall()

Save your team's final design to a JSON file. Give it a name you'll recognise later, e.g. including your team name.

In [ ]:
write_to_json_file("my_team_design.json", my_design)

In [ ]:
GLMakie.closeall()

## Discussion prep (for 15:00)

Jot down quick notes on the following — your team will share these in the discussion session:

1. Which parameter(s) did you change in your final design?
2. Did your first attempt break a requirement you weren't targeting? Which one, and why do you think that happened?
3. How much heavier or lighter is your final cell compared to the baseline (`my_design` mass vs. `cell_parameters` mass)?
4. If GreenWheels came back and asked for **20% lower cost** (roughly: less active material, thinner coatings), which requirement would you expect to sacrifice first?
5. Would you expect your design to still meet the brief on a hot day, or after 500 cycles? (You don't need to simulate this now — just discuss.)